In [7]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [8]:
# ── Core Libraries ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Scikit-learn ──
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, silhouette_score
)


In [9]:
# --- First Stape load an Clean data ---

#path = '/kaggle/input/datasets/mamadoulaminefallai/hakathon-week6/train.csv' # Kaggle path
path = r"C:\Users\bmd tech\Desktop\Spark Academy\Week6\SPARK-TEAM-2-SENEGAL\Hackaton week 6\data\train.csv"
train_df = pd.read_csv(f"{path}")      # chage path if run localy

train_df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
0,641,53,Male,Switzerland,non-anginal,160.0,0.0,NaN,lv hypertrophy,122.0,True,0.0,NaN,NaN,reversable defect,1
1,744,74,Male,VA Long Beach,non-anginal,NaN,0.0,False,normal,NaN,NaN,NaN,NaN,NaN,NaN,0
2,891,53,Male,VA Long Beach,asymptomatic,124.0,243.0,False,normal,122.0,True,2.0,flat,NaN,reversable defect,1
3,271,61,Male,Cleveland,asymptomatic,140.0,207.0,False,lv hypertrophy,138.0,True,1.9,upsloping,1.0,reversable defect,1
4,655,56,Male,Switzerland,non-anginal,155.0,0.0,False,st-t abnormality,99.0,False,0.0,flat,NaN,normal,1


In [10]:
train_df.tail()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
731,834,72,Male,VA Long Beach,asymptomatic,NaN,211.0,False,normal,NaN,NaN,NaN,NaN,NaN,NaN,1
732,243,49,Female,Cleveland,asymptomatic,130.0,269.0,False,normal,163.0,False,0.0,upsloping,0.0,normal,0
733,710,66,Male,Switzerland,asymptomatic,150.0,0.0,False,normal,108.0,True,2.0,flat,NaN,reversable defect,1
734,22,58,Female,Cleveland,typical angina,150.0,283.0,True,lv hypertrophy,162.0,False,1.0,upsloping,0.0,normal,0
735,318,35,Male,Hungary,atypical angina,120.0,308.0,False,lv hypertrophy,180.0,False,0.0,NaN,NaN,NaN,0


In [11]:
print(f"---- DataFrame Shape: {train_df.shape} ----\n")
print("="*60,f"Columns Name : {"="*60} \n {list(train_df.columns)}\n", "="*135)

---- DataFrame Shape: (736, 16) ----

============================================================ Columns Name : ============================================================ 
 ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


## Let's take some statistique to nown the dataset

In [12]:
# ── Basic data inspection ──
print("--- Data Types ---")
print(train_df.dtypes)
print(f"\n--- Missing Values ---")
print(train_df.isnull().sum())

train_df.columns.tolist()

--- Data Types ---
id            int64
age           int64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
ca          float64
thal         object
target        int64
dtype: object

--- Missing Values ---
id            0
age           0
sex           0
dataset       0
cp            0
trestbps     50
chol         27
fbs          68
restecg       0
thalch       47
exang        47
oldpeak      52
slope       241
ca          485
thal        381
target        0
dtype: int64


['id',
 'age',
 'sex',
 'dataset',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalch',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal',
 'target']

In [13]:
print(f"\n--- target Distribution ---")
print(train_df['target'].value_counts())
print(f"\nHeart disease prevalence: {train_df['target'].mean():.1%}")


--- target Distribution ---
target
1    407
0    329
Name: count, dtype: int64

Heart disease prevalence: 55.3%


In [14]:
# ── Check for impossible zero values (missing data coded as 0) ──
zero_cols = ['trestbps','chol','fbs','thalch','exang','oldpeak','slope','ca','thal'] 
print("--- Count of zero values ( missing data) ---")

for col in zero_cols:
    nbr_zeros = (train_df[col] == 0).sum()
    pct = nbr_zeros / len(train_df) * 100
    print(f"  {col:15s}: {nbr_zeros:8d} zeros ({pct:.1f}%)")

# ── Replace impossible zeros with NaN for cleaner analysis ──
print(f"\nTotal complete cases: {train_df.dropna().shape[0]} / {len(train_df)}")  # case complette sans zeros

train_df = train_df.copy()

for col in zero_cols:
   train_df[col] = train_df[col].replace(0, np.nan)

train_df.dropna(inplace=True)

--- Count of zero values ( missing data) ---
  trestbps       :        0 zeros (0.0%)
  chol           :      132 zeros (17.9%)
  fbs            :      560 zeros (76.1%)
  thalch         :        0 zeros (0.0%)
  exang          :      416 zeros (56.5%)
  oldpeak        :      288 zeros (39.1%)
  slope          :        0 zeros (0.0%)
  ca             :      151 zeros (20.5%)
  thal           :        0 zeros (0.0%)

Total complete cases: 246 / 736


In [15]:
train_df.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca,target
count,6.000000,6.00000,6.000000,6.000000,6.000000,6.000000,6.000000,6.0
mean,102.833333,59.50000,146.666667,263.500000,146.000000,1.666667,1.833333,1.0
std,72.628966,4.27785,33.856560,39.170142,13.725888,1.211060,0.752773,0.0
min,13.000000,56.00000,117.000000,228.000000,132.000000,0.600000,1.000000,1.0
25%,52.000000,56.00000,126.250000,234.750000,135.250000,1.050000,1.250000,1.0
50%,115.500000,58.00000,130.000000,252.500000,143.000000,1.300000,2.000000,1.0
75%,125.000000,62.25000,166.000000,280.000000,156.000000,1.700000,2.000000,1.0
max,214.000000,66.00000,200.000000,330.000000,165.000000,4.000000,3.000000,1.0


In [16]:
# Supprimer colonnes inutiles
train_df = train_df.drop(columns=["id", "dataset"])

# Encoder toutes les colonnes catégorielles automatiquement
X = pd.get_dummies(train_df.drop(columns=["target"]), drop_first=True)
y = train_df["target"]